# E1 on Kaggle: bge-small embeddings of `desc` → CatBoost (E0 vs E1)

Runs the full E0/E1 pipeline on a free Kaggle GPU (T4). The embedding step takes about 2 minutes
here, versus about 35 minutes on a laptop CPU. The same notebook also runs locally from `notebooks/`.

**Kaggle setup (right-hand panel)**
1. *Accelerator*: **GPU T4 x1**. *Internet*: **On** (to download `BAAI/bge-small-en-v1.5`).
2. *Add Input* → dataset **`wordsforthewise/lending-club`**.
3. Provide the code, either way:
   - zip this repo (without `data/`, `cache/`) and upload it as a private Kaggle Dataset,
     then add it as an input; **or**
   - set `GIT_URL` below to your repo.

**Outputs**: `/kaggle/working/e1_outputs.zip` contains the embeddings parquet, the embedding
cache, and `reports/`. Unzip it into the repo root locally, and `scripts.run_experiment` then
reuses the embeddings with no recomputation.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys
from pathlib import Path

GIT_URL = ""  # e.g. "https://github.com/<you>/llm-credit-risk-lab.git"; leave "" to use an uploaded dataset
ON_KAGGLE = Path("/kaggle/input").exists()

if ON_KAGGLE:
    REPO = Path("/kaggle/working/repo")
    if not REPO.exists():
        if GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            src = [
                Path(p).parent for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)
            ]
            assert src, "Code not found: upload the repo as a Kaggle Dataset or set GIT_URL"
            shutil.copytree(src[0], REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repo:", REPO, "| kaggle:", ON_KAGGLE)

In [ ]:
# Kaggle images already ship torch, sentence-transformers, catboost and pyarrow. This only
# fills gaps (quiet no-op when everything is present).
import importlib.util

missing = [
    p
    for p in ["catboost", "sentence_transformers", "pyarrow", "yaml"]
    if importlib.util.find_spec(p) is None
]
if missing:
    names = {"sentence_transformers": "sentence-transformers", "yaml": "pyyaml"}
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *[names.get(m, m) for m in missing]],
        check=True,
    )

import torch

print(
    "torch",
    torch.__version__,
    "| cuda:",
    torch.cuda.is_available(),
    "|",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
)

In [ ]:
def run(*args):
    """Run a repo CLI step and stream its output (same commands as on a laptop)."""
    print("$ python -m", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", *args], check=True)

## 1. Prepare data (skipped if the processed parquet exists)

In [ ]:
if not Path("data/processed/loans.parquet").exists():
    raw = sorted(
        glob.glob("/kaggle/input/**/accepted_2007_to_2018*.csv*", recursive=True)
        + glob.glob("data/raw/**/accepted_2007_to_2018*.csv*", recursive=True)
    )
    raw = [r for r in raw if Path(r).is_file()]
    assert raw, "Add the wordsforthewise/lending-club dataset as an input"
    run("scripts.prepare_data", "--config", "configs/base.yaml", "--raw", raw[0])
else:
    print("data/processed/loans.parquet exists, skipping")

## 2. Embed `desc` (GPU if available; cached, so re-running is free)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
run(
    "scripts.embed",
    "--config",
    "configs/e1.yaml",
    "--device",
    device,
    "--batch-size",
    "256" if device == "cuda" else "64",
)

## 3. Train E0 (baseline) and E1 (E0 + 32 embedding PCs), compare with a paired bootstrap

In [ ]:
if not Path("reports/runs/e0/predictions.parquet").exists():
    run("scripts.run_experiment", "--config", "configs/e0.yaml")
run("scripts.run_experiment", "--config", "configs/e1.yaml")

In [ ]:
import json

m0 = json.load(open("reports/runs/e0/metrics.json"))
m1 = json.load(open("reports/runs/e1/metrics.json"))
c = m1["vs_e0"]
row = lambda m: (
    f"{m['gini']['test']['value']:.4f} [{m['gini']['test']['ci_low']:.4f}, {m['gini']['test']['ci_high']:.4f}]"
)
print("E0 test Gini", row(m0))
print("E1 test Gini", row(m1))
g = c["gini_diff"]
print(
    f"E1 - E0: {g['diff']:+.4f} [{g['ci_low']:+.4f}, {g['ci_high']:+.4f}], P(diff<=0) = {g['p_value_one_sided']:.3f}"
)

In [ ]:
from IPython.display import Image, display

for f in ["e0_calibration", "e1_calibration", "e1_gini_by_quarter"]:
    display(Image(f"reports/figures/{f}.png", width=420))

## 4. Package outputs for download

In [ ]:
if ON_KAGGLE:
    bundle = Path("/kaggle/working/e1_bundle")
    shutil.rmtree(bundle, ignore_errors=True)
    for p in ["data/processed/embeddings", "cache/embeddings", "reports"]:
        shutil.copytree(p, bundle / p)
    print(shutil.make_archive("/kaggle/working/e1_outputs", "zip", bundle))